# Figure 4C: fit RMSE vs experimental range, implicit vs explicit

Per-solvent fit RMSE vs experimental shift range (¹H), implicit (PCM) vs explicit (Desmond).

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import delta22
import paths
import fig4_plots

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
# solvent_mean is a synthetic pseudo-solvent representing the average across solvents; it is used below as the reference baseline.
METHOD, BASIS, GEOM = "b3lyp_d3bj", "pcSseg2", "aimnet2"
q = delta22.load_query_df_dft(DELTA22_HDF5, XLSX, verbose=False)
one = q[(q["sap_nmr_method"] == METHOD) & (q["sap_basis"] == BASIS) & (q["sap_geometry_type"] == GEOM)]
one = delta22.add_solvent_mean(one)
print(len(one), "rows for", METHOD, BASIS, GEOM)

In [ ]:
FIG4C_LABELS = {
    "chloroform": "CDCl3", "dichloromethane": "DCM", "tetrahydrofuran": "THF",
    "acetonitrile": "MeCN", "dimethylsulfoxide": "DMSO", "acetone": "acetone",
    "methanol": "MeOD", "TIP4P": "TIP4P", "trifluoroethanol": "trifluoroethanol",
    "benzene": "benzene", "toluene": "toluene", "chlorobenzene": "chlorobenzene",
}
FIG4C_COLORS = {"Implicit (PCM)": "#A72608", "Explicit (Desmond)": "#61a89a"}  # red diamond / teal circle

In [ ]:
# differences are taken against the solvent-averaged pseudo-solvent so every real solvent, including chloroform, appears as a point.
rmse_range_rows = []
for solvent in delta22.DESMOND_SOLVENTS:
    sp = delta22.solvent_pair_differences(one, solvent, "solvent_mean", nucleus="H", explicit="desmond")
    if len(sp) == 0:
        continue
    rmse_range_rows.append({
        "solvent": solvent,
        "range": float(sp["exp_diff"].max() - sp["exp_diff"].min()),
        "implicit": delta22.fit_differences_to_experimental(sp, "implicit_diff")["rmse"],
        "explicit": delta22.fit_differences_to_experimental(sp, "explicit_diff")["rmse"],
    })
for r in sorted(rmse_range_rows, key=lambda r: r["range"]):
    print(f"{FIG4C_LABELS[r['solvent']]:16s} range={r['range']:.3f}  "
          f"implicit={r['implicit']:.3f}  explicit={r['explicit']:.3f}")

In [ ]:
fig4_plots.save_panel(
    lambda ax: fig4_plots.draw_rmse_dumbbell(ax, rmse_range_rows, FIG4C_LABELS, FIG4C_COLORS),
    figure_path("fig4c_rmse_range_1H.png"),
)